In [183]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

#PREPROCESSING:

####Read in raw training set:

In [184]:
DF = pd.read_csv('../data/raw/train.csv')
print(DF.head())

       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the process of invoking ...   
3  ["Creating a test set for a very rare category...   
4  ["The best way to travel from Tel Aviv to Jeru...   

 

####Clean & reformat dataset to just include required columns

In [185]:
longDF = pd.concat( [
    DF[['model_a', 'prompt', 'response_a']].rename(columns={'model_a': 'LLM_name', 'prompt': 'LLM_Input', 'response_a': 'LLM_output'}),
    DF[['model_b', 'prompt', 'response_b']].rename(columns={'model_b': 'LLM_name','prompt': 'LLM_Input', 'response_b': 'LLM_output'}),
], ignore_index=True)

print(longDF.head())

DF = longDF

             LLM_name                                          LLM_Input  \
0  gpt-4-1106-preview  ["Is it morally right to try to have a certain...   
1           koala-13b  ["What is the difference between marriage lice...   
2  gpt-3.5-turbo-0613  ["explain function calling. how would you call...   
3    llama-2-13b-chat  ["How can I create a test set for a very rare ...   
4           koala-13b  ["What is the best way to travel from Tel-Aviv...   

                                          LLM_output  
0  ["The question of whether it is morally right ...  
1  ["A marriage license is a legal document that ...  
2  ["Function calling is the process of invoking ...  
3  ["Creating a test set for a very rare category...  
4  ["The best way to travel from Tel Aviv to Jeru...  


####Get counts of each different model

In [186]:
print(DF["LLM_name"].value_counts().to_string())

LLM_name
gpt-4-1106-preview                7387
gpt-3.5-turbo-0613                7083
gpt-4-0613                        6165
claude-2.1                        5583
claude-instant-1                  4136
gpt-4-0314                        4122
claude-1                          3978
vicuna-33b                        3720
mixtral-8x7b-instruct-v0.1        3545
vicuna-13b                        3448
llama-2-70b-chat                  3428
gpt-3.5-turbo-1106                3352
mistral-medium                    3315
llama-2-13b-chat                  2607
claude-2.0                        2456
zephyr-7b-beta                    2401
palm-2                            1977
llama-2-7b-chat                   1793
wizardlm-70b                      1644
openchat-3.5                      1632
mistral-7b-instruct               1617
koala-13b                         1598
vicuna-7b                         1591
wizardlm-13b                      1580
oasst-pythia-12b                  1494
gemini-pro-dev-a

####Narrowed down to just 3 common LLMs: gpt-4-0613, claude-2.1, llama-2-70b-chat

In [187]:
DF = DF.loc[DF['LLM_name'].isin(['gpt-4-0613', 'claude-2.1', 'llama-2-70b-chat'])]
DF.head()

,LLM_name,LLM_Input,LLM_output
20,claude-2.1,"[""Write a python code that calculates sum of 5...","[""Here is the python code that calculates the ..."
38,gpt-4-0613,"[""Hi who are you""]","[""Hello! I'm an AI developed to assist with ta..."
45,gpt-4-0613,"[""write the lyrics to a rap song about some du...","[""(Verse 1)\nYo, let's take a minute, prepare ..."
47,claude-2.1,"[""#include <stdio.h>\n#include <stdlib.h>\n#in...","[""This C program analyzes the contents of a te..."
48,gpt-4-0613,"[""Write OCL statement each Employee working in...","[""context Employee inv:\nself.age > 22""]"


####Create testing/training split using StratifiedKFold to guarantee class proportions with 80/20 split

In [188]:
X = DF[['LLM_Input', 'LLM_output']]
Y = DF['LLM_name']

dataSplit = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(dataSplit.split(X, Y), start=1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y.iloc[train_idx]
    Y_test = Y.iloc[test_idx]

    print(f"Fold {fold}: {len(X_train)} train, {len(X_test)} test")
    

X_train.to_csv('../data/processed/XTrain.csv', index=False)
X_test.to_csv('../data/processed/XTest.csv', index=False)
Y_train.to_csv('../data/processed/YTrain.csv', index=False)
Y_test.to_csv('../data/processed/YTest.csv', index=False)

Fold 1: 12140 train, 3036 test
Fold 2: 12141 train, 3035 test
Fold 3: 12141 train, 3035 test
Fold 4: 12141 train, 3035 test
Fold 5: 12141 train, 3035 test


##CNN MODEL

In [189]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [190]:
import sys
import platform
import tensorflow as tf

print("Operating system:", platform.system())
print("Python executable:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

Operating system: Windows
Python executable: e:\python\python.exe
TensorFlow version: 2.21.0
Built with CUDA: False
GPUs: []


#### Read in data and OHE for target column

In [191]:
tf.keras.utils.set_random_seed(42)

X_train = pd.read_csv('../data/processed/XTrain.csv')
X_test = pd.read_csv('../data/processed/XTest.csv')
Y_train = pd.read_csv('../data/processed/YTrain.csv')
Y_test = pd.read_csv('../data/processed/YTest.csv')

X_train_Input = X_train['LLM_Input'].fillna("").astype(str).to_numpy()
X_train_Output = X_train['LLM_output'].fillna("").astype(str).to_numpy()

X_test_Input = X_test['LLM_Input'].fillna("").astype(str).to_numpy()
X_test_Output = X_test['LLM_output'].fillna("").astype(str).to_numpy()

X_train_Full = ("inputfield " + X_train["LLM_Input"].fillna("").astype(str) + " outputfield " + X_train["LLM_output"].fillna("").astype(str)).to_numpy()
X_test_Full = ("inputfield " + X_test["LLM_Input"].fillna("").astype(str) + " outputfield " + X_test["LLM_output"].fillna("").astype(str)).to_numpy()

encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.fit_transform(Y_test)




e:\python\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
e:\python\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


####Tokenizing LLM responses

In [192]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 1000

def createTokenizer(trainingSet):
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        standardize="lower_and_strip_punctuation",
        split="whitespace",
        output_mode="int",
        output_sequence_length=SEQUENCE_LENGTH,
    )

    # Fit the vocabulary ONLY on the training subset.
    vectorizer.adapt(
        tf.data.Dataset.from_tensor_slices(trainingSet).batch(64)
    )
    # lengths = pd.Series(trainingSet).str.split().str.len()
    # print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
    return vectorizer



In [193]:
def trainModelCNN(vectorizer, kernelSize, lr, filters):
    inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="text")

    x = vectorizer(inputs)

    x = tf.keras.layers.Embedding(
        input_dim=len(vectorizer.get_vocabulary()),
        output_dim=128,
    )(x)

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernelSize,
        activation="relu",
        padding="valid",
    )(x)

    x = tf.keras.layers.GlobalMaxPooling1D()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(
        3,
        activation="softmax",
    )(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # model.summary()
    return(model)

In [194]:
def trainModelRNN(vectorizer, enabledDense, lr, output_dim):
    inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="text")

    x = vectorizer(inputs)

    x = tf.keras.layers.Embedding(
        input_dim=len(vectorizer.get_vocabulary()),
        output_dim=output_dim,
        mask_zero=True,
    )(x)

    x = tf.keras.layers.LSTM(
        units=64,
        return_sequences=False
    )(x)

    x = tf.keras.layers.Dropout(0.5)(x)
    if (enabledDense):
        x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(
        3,
        activation="softmax",
    )(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # model.summary()
    return(model)

In [195]:
def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (texts.reshape(-1, 1), labels)
    )
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42,
            reshuffle_each_iteration=True,
        )
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

def applyModel(trainingSet, trainingOutput, validationSet, validationOutput, testingSet, testingOutput, model):

    train_ds = make_dataset(trainingSet, trainingOutput, shuffle=True)
    val_ds = make_dataset(validationSet, validationOutput)
    test_ds = make_dataset(testingSet, testingOutput)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
            )
        ],
    )
    test_loss, test_accuracy = model.evaluate(test_ds)

    probabilities = model.predict(test_ds)
    predicted_ids = probabilities.argmax(axis=1)

    print("Test accuracy:", test_accuracy)

    print(classification_report(
        Y_test,
        predicted_ids,
        labels=np.arange(3),
        target_names=[str(label) for label in encoder.classes_],
        zero_division=0,
    ))

    print(confusion_matrix(
        Y_test,
        predicted_ids,
        labels=np.arange(3),
    ))

In [196]:
def applyCNN(X_Train, X_Test, Y_Train, Y_Test):
    X_Train, Val_Train, Y_Train, Y_Val = train_test_split(
    X_Train,
    Y_Train,
    test_size=0.2,
    random_state=42, 
    stratify=Y_train)
    
    kernelSizes = [3, 5, 7]
    filters = [64, 128, 256]
    differentLrs = [3e-4, 1e-3, 3e-3]
    
    for i in kernelSizes:
        for j in filters:
            for k in differentLrs:
                print('--------------------------------------')
                print('CNN with:')
                print('Kernel size: ' + str(i))
                print('Filters: ' + str(j))
                print('Learning rate: ' + str(k))
                print('--------------------------------------')
                vectorizer = createTokenizer(X_Train)
                model = trainModelCNN(vectorizer, i, k, j)
                applyModel(X_Train, Y_Train, Val_Train, Y_Val, X_Test, Y_Test, model)
    
def applyRNN(X_Train, X_Test, Y_Train, Y_Test):
    X_Train, Val_Train, Y_Train, Y_Val = train_test_split(
    X_Train,
    Y_Train,
    test_size=0.2,
    random_state=42, 
    stratify=Y_train)
    
    vectorizer = createTokenizer(X_Train)
    
    enabledDense = [False, True]
    outputDims = [64, 128]
    differentLrs = [3e-4, 3e-3]
    
    for i in enabledDense:
        for j in outputDims:
            for k in differentLrs:
                print('--------------------------------------')
                print('RNN with:')
                print('Hidden dense layer: ' + str(i))
                print('Output dimensions: ' + str(j))
                print('Learning rate: ' + str(k))
                print('--------------------------------------')
                model = trainModelRNN(vectorizer, i, k, j)
                applyModel(X_Train, Y_Train, Val_Train, Y_Val, X_Test, Y_Test, model)

In [197]:
applyCNN(X_train_Output, X_test_Output, Y_train, Y_test)


--------------------------------------
CNN with:
Kernel size: 3
Filters: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.5084 - loss: 1.0158 - val_accuracy: 0.6476 - val_loss: 0.8683
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.6549 - loss: 0.7906 - val_accuracy: 0.7353 - val_loss: 0.6384
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.7271 - loss: 0.6321 - val_accuracy: 0.7830 - val_loss: 0.5606
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.7796 - loss: 0.5313 - val_accuracy: 0.7983 - val_loss: 0.5014
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.8311 - loss: 0.4293 - val_accuracy: 0.8114 - val_loss: 0.4608
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.8646 - loss: 0.3594 - val_accuracy: 0.8123 - val_loss: 0.4385
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.8994 - loss: 0.2824 - val_accuracy: 0.8152 - val_loss: 0.4280
Epoch 8/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.9202 - loss: 0.2259 - val_accuracy: 0.82

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.5915 - loss: 0.8653 - val_accuracy: 0.7666 - val_loss: 0.5794
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.7521 - loss: 0.5721 - val_accuracy: 0.8003 - val_loss: 0.4832
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8359 - loss: 0.4042 - val_accuracy: 0.8131 - val_loss: 0.4530
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.8967 - loss: 0.2805 - val_accuracy: 0.8189 - val_loss: 0.4502
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.9261 - loss: 0.2012 - val_accuracy: 0.8230 - val_loss: 0.4838
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.9493 - loss: 0.1451 - val_accuracy: 0.8217 - val_loss: 0.5002
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.9585 - loss: 0.1196 - val_accuracy: 0.8106 - val_loss: 0.5588
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8208 - loss: 0.4537
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 153ms/step - accuracy: 0.5070 - loss: 0.9957 - val_accuracy: 0.6542 - val_loss: 0.7965
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 153ms/step - accuracy: 0.7119 - loss: 0.6548 - val_accuracy: 0.7723 - val_loss: 0.5375
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 153ms/step - accuracy: 0.8082 - loss: 0.4456 - val_accuracy: 0.8156 - val_loss: 0.4701
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 153ms/step - accuracy: 0.8912 - loss: 0.2903 - val_accuracy: 0.8320 - val_loss: 0.4289
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 154ms/step - accuracy: 0.9436 - loss: 0.1750 - val_accuracy: 0.8357 - val_loss: 0.4218
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 154ms/step - accuracy: 0.9716 - loss: 0.0982 - val_accuracy: 0.8415 - val_loss: 0.4413
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 154ms/step - accuracy: 0.9827 - loss: 0.0624 - val_accuracy: 0.8349 - val_loss: 0.4927
Epoch 8/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 155ms/step - accuracy: 0.9881 - loss: 0.0417 - val

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 150ms/step - accuracy: 0.6324 - loss: 0.7700 - val_accuracy: 0.7884 - val_loss: 0.5127
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 150ms/step - accuracy: 0.8389 - loss: 0.3939 - val_accuracy: 0.8345 - val_loss: 0.4196
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 150ms/step - accuracy: 0.9328 - loss: 0.1807 - val_accuracy: 0.8275 - val_loss: 0.5009
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 153ms/step - accuracy: 0.9704 - loss: 0.0874 - val_accuracy: 0.8399 - val_loss: 0.5339
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 150ms/step - accuracy: 0.9845 - loss: 0.0462 - val_accuracy: 0.8361 - val_loss: 0.6603
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.8227 - loss: 0.4453
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step
Test accuracy: 0.8227347731590271
                  precision    recall  f1-score   support

      claude-2.1       0.95      0.84      0.89      1116
      gpt-4-0613       0.74      0.92      0.82      1233
llama-2-70b-chat       0.83      

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 46s 150ms/step - accuracy: 0.6836 - loss: 0.6825 - val_accuracy: 0.8254 - val_loss: 0.4471
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 45s 148ms/step - accuracy: 0.8630 - loss: 0.3490 - val_accuracy: 0.8304 - val_loss: 0.4358
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 45s 149ms/step - accuracy: 0.9363 - loss: 0.1765 - val_accuracy: 0.8152 - val_loss: 0.5113
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 45s 149ms/step - accuracy: 0.9575 - loss: 0.1171 - val_accuracy: 0.8077 - val_loss: 0.6541
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 45s 149ms/step - accuracy: 0.9677 - loss: 0.0911 - val_accuracy: 0.8016 - val_loss: 0.6643
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.8208 - loss: 0.4717
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step
Test accuracy: 0.8207578063011169
                  precision    recall  f1-score   support

      claude-2.1       0.89      0.88      0.88      1116
      gpt-4-0613       0.77      0.87      0.82      1233
llama-2-70b-chat       0.82      

In [198]:
applyRNN(X_train_Output, X_test_Output, Y_train, Y_test)

--------------------------------------
RNN with:
Hidden dense layer: False
Output dimensions: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 133s 431ms/step - accuracy: 0.5317 - loss: 0.9682 - val_accuracy: 0.6715 - val_loss: 0.7639
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 130s 426ms/step - accuracy: 0.7053 - loss: 0.6710 - val_accuracy: 0.6772 - val_loss: 0.7914
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 426ms/step - accuracy: 0.8194 - loss: 0.4885 - val_accuracy: 0.7608 - val_loss: 0.5822
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 424ms/step - accuracy: 0.9026 - loss: 0.2857 - val_accuracy: 0.7509 - val_loss: 0.6474
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 424ms/step - accuracy: 0.9347 - loss: 0.2021 - val_accuracy: 0.7876 - val_loss: 0.5852
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 424ms/step - accuracy: 0.9459 - loss: 0.1799 - val_accuracy: 0.7900 - val_loss: 0.5960
95/95 ━━━━━━━━━━━━━━━━━━━━ 13s 135ms/step - accuracy: 0.7400 - loss: 0.6101
95/95 ━━━━━━━━━━━━━━━━━━━━ 13s 136ms/step
Test accuracy: 0.740032970905304
                  precision    recall  f1-score   support

      cla

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 419ms/step - accuracy: 0.5764 - loss: 0.8973 - val_accuracy: 0.6583 - val_loss: 0.7891
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 127s 417ms/step - accuracy: 0.7716 - loss: 0.5713 - val_accuracy: 0.5488 - val_loss: 0.9548
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 127s 418ms/step - accuracy: 0.8658 - loss: 0.3740 - val_accuracy: 0.6690 - val_loss: 0.9577
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 128s 420ms/step - accuracy: 0.9381 - loss: 0.1867 - val_accuracy: 0.6871 - val_loss: 1.0955
95/95 ━━━━━━━━━━━━━━━━━━━━ 12s 131ms/step - accuracy: 0.6623 - loss: 0.7659
95/95 ━━━━━━━━━━━━━━━━━━━━ 13s 132ms/step
Test accuracy: 0.6622734665870667
                  precision    recall  f1-score   support

      claude-2.1       0.88      0.69      0.77      1116
      gpt-4-0613       0.57      0.89      0.69      1233
llama-2-70b-chat       0.60      0.22      0.32       686

        accuracy                           0.66      3035
       macro avg       0.68      0.60  

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 117s 380ms/step - accuracy: 0.4982 - loss: 0.9998 - val_accuracy: 0.5138 - val_loss: 1.0004
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 115s 379ms/step - accuracy: 0.6910 - loss: 0.7227 - val_accuracy: 0.6657 - val_loss: 0.7951
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 116s 381ms/step - accuracy: 0.8322 - loss: 0.4309 - val_accuracy: 0.7007 - val_loss: 0.8096
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 116s 383ms/step - accuracy: 0.8974 - loss: 0.2861 - val_accuracy: 0.6995 - val_loss: 0.9279
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 115s 378ms/step - accuracy: 0.9545 - loss: 0.1324 - val_accuracy: 0.7048 - val_loss: 1.0973
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.6550 - loss: 0.7796
95/95 ━━━━━━━━━━━━━━━━━━━━ 11s 109ms/step
Test accuracy: 0.6550247073173523
                  precision    recall  f1-score   support

      claude-2.1       0.82      0.73      0.77      1116
      gpt-4-0613       0.59      0.81      0.68      1233
llama-2-70b-chat       0

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 86s 275ms/step - accuracy: 0.5036 - loss: 0.9906 - val_accuracy: 0.6501 - val_loss: 0.7960
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 83s 274ms/step - accuracy: 0.7512 - loss: 0.5808 - val_accuracy: 0.7822 - val_loss: 0.5592
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 83s 274ms/step - accuracy: 0.8298 - loss: 0.4713 - val_accuracy: 0.7089 - val_loss: 0.6749
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 83s 273ms/step - accuracy: 0.9040 - loss: 0.2733 - val_accuracy: 0.7711 - val_loss: 0.6394
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 83s 272ms/step - accuracy: 0.9500 - loss: 0.1491 - val_accuracy: 0.7797 - val_loss: 0.7150
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.7690 - loss: 0.5647
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 70ms/step
Test accuracy: 0.7690280079841614
                  precision    recall  f1-score   support

      claude-2.1       0.87      0.82      0.85      1116
      gpt-4-0613       0.70      0.83      0.76      1233
llama-2-70b-chat       0.75      

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 82s 266ms/step - accuracy: 0.5797 - loss: 0.8962 - val_accuracy: 0.6595 - val_loss: 0.7887
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 92s 304ms/step - accuracy: 0.7665 - loss: 0.5542 - val_accuracy: 0.7229 - val_loss: 0.6705
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 94s 308ms/step - accuracy: 0.8944 - loss: 0.2864 - val_accuracy: 0.7126 - val_loss: 0.8381
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 94s 309ms/step - accuracy: 0.9389 - loss: 0.1759 - val_accuracy: 0.6958 - val_loss: 1.0680
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 93s 307ms/step - accuracy: 0.9662 - loss: 0.1005 - val_accuracy: 0.7205 - val_loss: 1.1424
95/95 ━━━━━━━━━━━━━━━━━━━━ 8s 85ms/step - accuracy: 0.7163 - loss: 0.6816
95/95 ━━━━━━━━━━━━━━━━━━━━ 8s 86ms/step
Test accuracy: 0.7163097262382507
                  precision    recall  f1-score   support

      claude-2.1       0.84      0.79      0.82      1116
      gpt-4-0613       0.68      0.76      0.71      1233
llama-2-70b-chat       0.59      

In [199]:
applyCNN(X_train_Input, X_test_Input, Y_train, Y_test)

--------------------------------------
CNN with:
Kernel size: 3
Filters: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.3994 - loss: 1.0738 - val_accuracy: 0.4063 - val_loss: 1.0698
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.4215 - loss: 1.0614 - val_accuracy: 0.3821 - val_loss: 1.0692
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.4636 - loss: 1.0404 - val_accuracy: 0.3821 - val_loss: 1.0717
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.5329 - loss: 0.9885 - val_accuracy: 0.3783 - val_loss: 1.0846
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.6247 - loss: 0.8776 - val_accuracy: 0.3598 - val_loss: 1.1348
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4003 - loss: 1.0695
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Test accuracy: 0.40032950043678284
                  precision    recall  f1-score   support

      claude-2.1       0.38      0.19      0.25      1116
      gpt-4-0613       0.41      0.82      0.54      1233
llama-2-70b-chat       0.00      0.00      

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.4043 - loss: 1.0728 - val_accuracy: 0.3969 - val_loss: 1.0721
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.4320 - loss: 1.0532 - val_accuracy: 0.3751 - val_loss: 1.0720
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.5096 - loss: 0.9846 - val_accuracy: 0.3532 - val_loss: 1.1193
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.6298 - loss: 0.8330 - val_accuracy: 0.3602 - val_loss: 1.2111
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.7249 - loss: 0.6830 - val_accuracy: 0.3359 - val_loss: 1.3623
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3911 - loss: 1.0714
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Test accuracy: 0.3911037743091583
                  precision    recall  f1-score   support

      claude-2.1       0.37      0.39      0.38      1116
      gpt-4-0613       0.40      0.60      0.48      1233
llama-2-70b-chat       0.40      0.00      0

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 54s 176ms/step - accuracy: 0.3980 - loss: 1.0731 - val_accuracy: 0.4063 - val_loss: 1.0725
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 53s 173ms/step - accuracy: 0.4401 - loss: 1.0575 - val_accuracy: 0.3783 - val_loss: 1.0700
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 52s 172ms/step - accuracy: 0.4933 - loss: 1.0166 - val_accuracy: 0.3767 - val_loss: 1.0884
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 52s 173ms/step - accuracy: 0.5995 - loss: 0.8915 - val_accuracy: 0.3870 - val_loss: 1.1576
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 53s 174ms/step - accuracy: 0.7325 - loss: 0.6810 - val_accuracy: 0.3532 - val_loss: 1.3210
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.3812 - loss: 1.0698
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step
Test accuracy: 0.38121911883354187
                  precision    recall  f1-score   support

      claude-2.1       0.37      0.50      0.43      1116
      gpt-4-0613       0.39      0.48      0.43      1233
llama-2-70b-chat       0.00     

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 54s 176ms/step - accuracy: 0.3975 - loss: 1.0740 - val_accuracy: 0.3895 - val_loss: 1.0733
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 163ms/step - accuracy: 0.4467 - loss: 1.0473 - val_accuracy: 0.3903 - val_loss: 1.0799
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 156ms/step - accuracy: 0.5870 - loss: 0.9006 - val_accuracy: 0.3573 - val_loss: 1.1784
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 156ms/step - accuracy: 0.7408 - loss: 0.6548 - val_accuracy: 0.3466 - val_loss: 1.3557
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.4096 - loss: 1.0733
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step
Test accuracy: 0.40955519676208496
                  precision    recall  f1-score   support

      claude-2.1       0.41      0.13      0.19      1116
      gpt-4-0613       0.41      0.89      0.56      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.41      3035
       macro avg       0.27      0.34      0.2

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 48s 155ms/step - accuracy: 0.3927 - loss: 1.0747 - val_accuracy: 0.4014 - val_loss: 1.0721
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 155ms/step - accuracy: 0.4177 - loss: 1.0495 - val_accuracy: 0.3882 - val_loss: 1.0805
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 154ms/step - accuracy: 0.5149 - loss: 0.9420 - val_accuracy: 0.3660 - val_loss: 1.1410
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 47s 155ms/step - accuracy: 0.6421 - loss: 0.7687 - val_accuracy: 0.3594 - val_loss: 1.2827
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.4102 - loss: 1.0738
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step
Test accuracy: 0.4102141559123993
                  precision    recall  f1-score   support

      claude-2.1       0.42      0.18      0.25      1116
      gpt-4-0613       0.41      0.85      0.55      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.41      3035
       macro avg       0.28      0.34      0.27

In [200]:
applyRNN(X_train_Input, X_test_Input, Y_train, Y_test)

--------------------------------------
RNN with:
Hidden dense layer: False
Output dimensions: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 131s 428ms/step - accuracy: 0.3945 - loss: 1.0769 - val_accuracy: 0.4063 - val_loss: 1.0708
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 129s 425ms/step - accuracy: 0.4159 - loss: 1.0630 - val_accuracy: 0.3812 - val_loss: 1.0709
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 135s 445ms/step - accuracy: 0.4667 - loss: 1.0243 - val_accuracy: 0.3664 - val_loss: 1.0991
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 138s 453ms/step - accuracy: 0.5929 - loss: 0.8864 - val_accuracy: 0.3471 - val_loss: 1.1880
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 146ms/step - accuracy: 0.4059 - loss: 1.0719
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 145ms/step
Test accuracy: 0.4059308171272278
                  precision    recall  f1-score   support

      claude-2.1       0.00      0.00      0.00      1116
      gpt-4-0613       0.41      1.00      0.58      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.41      3035
       macro avg       0.14      0.33  

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 155s 502ms/step - accuracy: 0.3974 - loss: 1.0738 - val_accuracy: 0.4006 - val_loss: 1.0732
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 153s 504ms/step - accuracy: 0.4894 - loss: 1.0059 - val_accuracy: 0.3573 - val_loss: 1.1392
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 153s 503ms/step - accuracy: 0.6581 - loss: 0.7685 - val_accuracy: 0.3483 - val_loss: 1.3201
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 155s 509ms/step - accuracy: 0.7636 - loss: 0.5561 - val_accuracy: 0.3388 - val_loss: 1.5998
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 157ms/step - accuracy: 0.3990 - loss: 1.0732
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 154ms/step
Test accuracy: 0.3990115225315094
                  precision    recall  f1-score   support

      claude-2.1       0.36      0.13      0.19      1116
      gpt-4-0613       0.41      0.87      0.55      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.40      3035
       macro avg       0.25      0.33  

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 138s 449ms/step - accuracy: 0.3983 - loss: 1.0726 - val_accuracy: 0.4010 - val_loss: 1.0738
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 137s 452ms/step - accuracy: 0.4504 - loss: 1.0348 - val_accuracy: 0.3915 - val_loss: 1.1281
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 137s 451ms/step - accuracy: 0.6082 - loss: 0.8459 - val_accuracy: 0.3368 - val_loss: 1.2838
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 138s 454ms/step - accuracy: 0.7279 - loss: 0.6228 - val_accuracy: 0.3516 - val_loss: 1.5796
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 144ms/step - accuracy: 0.4020 - loss: 1.0733
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 145ms/step
Test accuracy: 0.40197694301605225
                  precision    recall  f1-score   support

      claude-2.1       0.35      0.05      0.09      1116
      gpt-4-0613       0.41      0.94      0.57      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.40      3035
       macro avg       0.25      0.33 

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 103s 333ms/step - accuracy: 0.4003 - loss: 1.0757 - val_accuracy: 0.4063 - val_loss: 1.0699
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 102s 335ms/step - accuracy: 0.4155 - loss: 1.0560 - val_accuracy: 0.3890 - val_loss: 1.0771
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 103s 338ms/step - accuracy: 0.5341 - loss: 0.9533 - val_accuracy: 0.3590 - val_loss: 1.1786
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 101s 332ms/step - accuracy: 0.6625 - loss: 0.7564 - val_accuracy: 0.3450 - val_loss: 1.3425
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - accuracy: 0.4063 - loss: 1.0702
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step
Test accuracy: 0.40626028180122375
                  precision    recall  f1-score   support

      claude-2.1       0.00      0.00      0.00      1116
      gpt-4-0613       0.41      1.00      0.58      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.41      3035
       macro avg       0.14      0.33     

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 108s 346ms/step - accuracy: 0.4003 - loss: 1.0730 - val_accuracy: 0.3985 - val_loss: 1.0731
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 106s 349ms/step - accuracy: 0.4653 - loss: 1.0292 - val_accuracy: 0.3882 - val_loss: 1.1299
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 104s 344ms/step - accuracy: 0.6121 - loss: 0.8308 - val_accuracy: 0.3347 - val_loss: 1.2989
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 102s 337ms/step - accuracy: 0.7428 - loss: 0.5829 - val_accuracy: 0.3421 - val_loss: 1.5946
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.4003 - loss: 1.0721
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step
Test accuracy: 0.40032950043678284
                  precision    recall  f1-score   support

      claude-2.1       0.31      0.03      0.06      1116
      gpt-4-0613       0.40      0.95      0.57      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.40      3035
       macro avg       0.24      0.33     

In [201]:
applyCNN(X_train_Full, X_test_Full, Y_train, Y_test)

--------------------------------------
CNN with:
Kernel size: 3
Filters: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.4912 - loss: 1.0157 - val_accuracy: 0.6781 - val_loss: 0.8491
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.6662 - loss: 0.7678 - val_accuracy: 0.7620 - val_loss: 0.6184
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.7290 - loss: 0.6305 - val_accuracy: 0.7867 - val_loss: 0.5533
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.7782 - loss: 0.5299 - val_accuracy: 0.7950 - val_loss: 0.5081
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.8269 - loss: 0.4358 - val_accuracy: 0.8049 - val_loss: 0.4755
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.8662 - loss: 0.3518 - val_accuracy: 0.8135 - val_loss: 0.4542
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.8980 - loss: 0.2800 - val_accuracy: 0.8147 - val_loss: 0.4506
Epoch 8/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9229 - loss: 0.2172 - val_accurac

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.5965 - loss: 0.8599 - val_accuracy: 0.7542 - val_loss: 0.6038
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.7481 - loss: 0.5889 - val_accuracy: 0.7818 - val_loss: 0.5008
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.8323 - loss: 0.4248 - val_accuracy: 0.8094 - val_loss: 0.4721
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9028 - loss: 0.2747 - val_accuracy: 0.8168 - val_loss: 0.4688
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9329 - loss: 0.1964 - val_accuracy: 0.8094 - val_loss: 0.5027
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9516 - loss: 0.1383 - val_accuracy: 0.8164 - val_loss: 0.5335
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9614 - loss: 0.1107 - val_accuracy: 0.8127 - val_loss: 0.5778
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8000 - loss: 0.4971
95/95 ━━━━━━━━━━━━━━━━━━━━ 1

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 50s 161ms/step - accuracy: 0.5255 - loss: 0.9630 - val_accuracy: 0.6855 - val_loss: 0.7069
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 160ms/step - accuracy: 0.7302 - loss: 0.6169 - val_accuracy: 0.7686 - val_loss: 0.5372
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 160ms/step - accuracy: 0.8254 - loss: 0.4267 - val_accuracy: 0.8020 - val_loss: 0.4775
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 52s 170ms/step - accuracy: 0.9017 - loss: 0.2694 - val_accuracy: 0.8143 - val_loss: 0.4592
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 161ms/step - accuracy: 0.9544 - loss: 0.1491 - val_accuracy: 0.8197 - val_loss: 0.4790
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 161ms/step - accuracy: 0.9822 - loss: 0.0752 - val_accuracy: 0.8106 - val_loss: 0.5109
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 160ms/step - accuracy: 0.9896 - loss: 0.0447 - val_accuracy: 0.8110 - val_loss: 0.5765
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.8092 - loss: 0.4602
95/95 ━━━━━━━━━━━━━━

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 160ms/step - accuracy: 0.6416 - loss: 0.7633 - val_accuracy: 0.7867 - val_loss: 0.5271
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 161ms/step - accuracy: 0.8374 - loss: 0.4024 - val_accuracy: 0.8283 - val_loss: 0.4307
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 161ms/step - accuracy: 0.9422 - loss: 0.1601 - val_accuracy: 0.8172 - val_loss: 0.5433
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 160ms/step - accuracy: 0.9790 - loss: 0.0691 - val_accuracy: 0.8254 - val_loss: 0.5970
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 53s 174ms/step - accuracy: 0.9895 - loss: 0.0368 - val_accuracy: 0.8213 - val_loss: 0.6819
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.8257 - loss: 0.4508
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step
Test accuracy: 0.8257001638412476
                  precision    recall  f1-score   support

      claude-2.1       0.94      0.84      0.89      1116
      gpt-4-0613       0.76      0.91      0.82      1233
llama-2-70b-chat       0.80      

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 51s 164ms/step - accuracy: 0.6814 - loss: 0.6822 - val_accuracy: 0.8040 - val_loss: 0.4860
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 49s 163ms/step - accuracy: 0.8614 - loss: 0.3550 - val_accuracy: 0.8156 - val_loss: 0.4626
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 52s 170ms/step - accuracy: 0.9436 - loss: 0.1553 - val_accuracy: 0.7888 - val_loss: 0.6358
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 51s 168ms/step - accuracy: 0.9647 - loss: 0.1024 - val_accuracy: 0.8053 - val_loss: 0.6820
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 51s 167ms/step - accuracy: 0.9801 - loss: 0.0609 - val_accuracy: 0.8090 - val_loss: 0.6857
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.8145 - loss: 0.4744
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step
Test accuracy: 0.8144975304603577
                  precision    recall  f1-score   support

      claude-2.1       0.89      0.87      0.88      1116
      gpt-4-0613       0.81      0.80      0.80      1233
llama-2-70b-chat       0.71      

In [202]:
applyRNN(X_train_Full, X_test_Full, Y_train, Y_test)

--------------------------------------
RNN with:
Hidden dense layer: False
Output dimensions: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 101s 327ms/step - accuracy: 0.4865 - loss: 1.0282 - val_accuracy: 0.5772 - val_loss: 0.8851
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 325ms/step - accuracy: 0.6501 - loss: 0.7796 - val_accuracy: 0.7110 - val_loss: 0.6805
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 98s 322ms/step - accuracy: 0.8245 - loss: 0.4573 - val_accuracy: 0.7612 - val_loss: 0.5933
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 325ms/step - accuracy: 0.9017 - loss: 0.2819 - val_accuracy: 0.7563 - val_loss: 0.6325
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 98s 323ms/step - accuracy: 0.9352 - loss: 0.2117 - val_accuracy: 0.7674 - val_loss: 0.6833
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 98s 323ms/step - accuracy: 0.9435 - loss: 0.1764 - val_accuracy: 0.7604 - val_loss: 0.7333
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - accuracy: 0.7562 - loss: 0.5949
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step
Test accuracy: 0.7561779022216797
                  precision    recall  f1-score   support

      claude-

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 98s 318ms/step - accuracy: 0.5467 - loss: 0.9591 - val_accuracy: 0.6085 - val_loss: 0.8587
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 96s 317ms/step - accuracy: 0.7451 - loss: 0.6159 - val_accuracy: 0.6826 - val_loss: 0.7546
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 96s 317ms/step - accuracy: 0.8760 - loss: 0.3397 - val_accuracy: 0.7028 - val_loss: 0.8489
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 96s 315ms/step - accuracy: 0.9371 - loss: 0.1935 - val_accuracy: 0.6925 - val_loss: 1.0510
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 96s 315ms/step - accuracy: 0.9676 - loss: 0.1023 - val_accuracy: 0.6896 - val_loss: 1.1293
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 99ms/step - accuracy: 0.6751 - loss: 0.7387
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step
Test accuracy: 0.6751235723495483
                  precision    recall  f1-score   support

      claude-2.1       0.78      0.80      0.79      1116
      gpt-4-0613       0.68      0.65      0.66      1233
llama-2-70b-chat       0.50     

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 102s 330ms/step - accuracy: 0.5581 - loss: 0.9147 - val_accuracy: 0.6731 - val_loss: 0.7420
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 101s 332ms/step - accuracy: 0.7688 - loss: 0.5640 - val_accuracy: 0.6612 - val_loss: 0.7941
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 102s 335ms/step - accuracy: 0.8880 - loss: 0.2983 - val_accuracy: 0.6888 - val_loss: 0.9612
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 327ms/step - accuracy: 0.9471 - loss: 0.1563 - val_accuracy: 0.6447 - val_loss: 1.3523
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - accuracy: 0.6738 - loss: 0.7291
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step
Test accuracy: 0.6738055944442749
                  precision    recall  f1-score   support

      claude-2.1       0.85      0.75      0.79      1116
      gpt-4-0613       0.60      0.78      0.68      1233
llama-2-70b-chat       0.54      0.36      0.43       686

        accuracy                           0.67      3035
       macro avg       0.66      0.63   

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 71s 230ms/step - accuracy: 0.5194 - loss: 0.9852 - val_accuracy: 0.6451 - val_loss: 0.7785
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 230ms/step - accuracy: 0.7423 - loss: 0.5965 - val_accuracy: 0.7538 - val_loss: 0.5941
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 231ms/step - accuracy: 0.8833 - loss: 0.3247 - val_accuracy: 0.7600 - val_loss: 0.6145
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 231ms/step - accuracy: 0.9398 - loss: 0.1843 - val_accuracy: 0.7563 - val_loss: 0.7671
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 232ms/step - accuracy: 0.9598 - loss: 0.1269 - val_accuracy: 0.7361 - val_loss: 0.8788
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.7496 - loss: 0.5960
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 75ms/step
Test accuracy: 0.749588131904602
                  precision    recall  f1-score   support

      claude-2.1       0.80      0.83      0.81      1116
      gpt-4-0613       0.72      0.76      0.74      1233
llama-2-70b-chat       0.71      0

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 72s 232ms/step - accuracy: 0.5800 - loss: 0.8929 - val_accuracy: 0.6365 - val_loss: 0.8341
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 232ms/step - accuracy: 0.7891 - loss: 0.5191 - val_accuracy: 0.6929 - val_loss: 0.7340
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 232ms/step - accuracy: 0.9126 - loss: 0.2448 - val_accuracy: 0.6978 - val_loss: 0.8876
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 71s 232ms/step - accuracy: 0.9575 - loss: 0.1264 - val_accuracy: 0.6933 - val_loss: 1.1436
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 70s 232ms/step - accuracy: 0.9789 - loss: 0.0733 - val_accuracy: 0.7011 - val_loss: 1.3035
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.6814 - loss: 0.7349
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 76ms/step
Test accuracy: 0.6813838481903076
                  precision    recall  f1-score   support

      claude-2.1       0.81      0.76      0.78      1116
      gpt-4-0613       0.65      0.70      0.67      1233
llama-2-70b-chat       0.55      

####Testing with different hyper params for tokenization

In [203]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 5000

In [204]:
applyCNN(X_train_Full, X_test_Full, Y_train, Y_test)

--------------------------------------
CNN with:
Kernel size: 3
Filters: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 43s 138ms/step - accuracy: 0.5170 - loss: 1.0074 - val_accuracy: 0.6521 - val_loss: 0.8471
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.6586 - loss: 0.7739 - val_accuracy: 0.7398 - val_loss: 0.6247
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.7242 - loss: 0.6314 - val_accuracy: 0.7682 - val_loss: 0.5715
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 43s 141ms/step - accuracy: 0.7759 - loss: 0.5382 - val_accuracy: 0.7921 - val_loss: 0.5156
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 43s 141ms/step - accuracy: 0.8249 - loss: 0.4434 - val_accuracy: 0.8036 - val_loss: 0.4797
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 139ms/step - accuracy: 0.8716 - loss: 0.3575 - val_accuracy: 0.8135 - val_loss: 0.4524
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 139ms/step - accuracy: 0.9076 - loss: 0.2740 - val_accuracy: 0.8184 - val_loss: 0.4429
Epoch 8/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.9276 - loss: 0.2197 - val

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 43s 139ms/step - accuracy: 0.5682 - loss: 0.9006 - val_accuracy: 0.7406 - val_loss: 0.6142
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 139ms/step - accuracy: 0.7538 - loss: 0.5867 - val_accuracy: 0.7888 - val_loss: 0.4916
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.8401 - loss: 0.4083 - val_accuracy: 0.8127 - val_loss: 0.4692
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 139ms/step - accuracy: 0.9032 - loss: 0.2706 - val_accuracy: 0.8090 - val_loss: 0.4755
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.9401 - loss: 0.1763 - val_accuracy: 0.8143 - val_loss: 0.5287
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 42s 138ms/step - accuracy: 0.9501 - loss: 0.1430 - val_accuracy: 0.8057 - val_loss: 0.5703
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.8003 - loss: 0.4907
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step
Test accuracy: 0.8003295063972473
                  precision    recall  f1-score   support

      claude-2.1  

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 860ms/step - accuracy: 0.5501 - loss: 0.9399 - val_accuracy: 0.7192 - val_loss: 0.6738
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 863ms/step - accuracy: 0.7420 - loss: 0.5986 - val_accuracy: 0.7744 - val_loss: 0.5243
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 862ms/step - accuracy: 0.8317 - loss: 0.4184 - val_accuracy: 0.8073 - val_loss: 0.4668
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 862ms/step - accuracy: 0.9101 - loss: 0.2544 - val_accuracy: 0.8226 - val_loss: 0.4467
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 263s 866ms/step - accuracy: 0.9631 - loss: 0.1345 - val_accuracy: 0.8250 - val_loss: 0.4699
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 863ms/step - accuracy: 0.9845 - loss: 0.0678 - val_accuracy: 0.8217 - val_loss: 0.5116
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 261s 859ms/step - accuracy: 0.9928 - loss: 0.0358 - val_accuracy: 0.8168 - val_loss: 0.5758
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 161ms/step - accuracy: 0.8089 - loss: 0.4720
95/95 ━━━━━

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 264s 865ms/step - accuracy: 0.6512 - loss: 0.7549 - val_accuracy: 0.7851 - val_loss: 0.5191
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 261s 858ms/step - accuracy: 0.8447 - loss: 0.3908 - val_accuracy: 0.8296 - val_loss: 0.4369
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 261s 857ms/step - accuracy: 0.9476 - loss: 0.1490 - val_accuracy: 0.8213 - val_loss: 0.5456
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 862ms/step - accuracy: 0.9818 - loss: 0.0589 - val_accuracy: 0.8242 - val_loss: 0.6035
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 863ms/step - accuracy: 0.9908 - loss: 0.0319 - val_accuracy: 0.8184 - val_loss: 0.6633
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 161ms/step - accuracy: 0.8086 - loss: 0.4711
95/95 ━━━━━━━━━━━━━━━━━━━━ 16s 166ms/step
Test accuracy: 0.8085667490959167
                  precision    recall  f1-score   support

      claude-2.1       0.92      0.84      0.88      1116
      gpt-4-0613       0.74      0.89      0.81      1233
llama-2-70b-chat       0

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 263s 863ms/step - accuracy: 0.6871 - loss: 0.6788 - val_accuracy: 0.8065 - val_loss: 0.4753
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 261s 859ms/step - accuracy: 0.8626 - loss: 0.3440 - val_accuracy: 0.8201 - val_loss: 0.4688
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 260s 856ms/step - accuracy: 0.9474 - loss: 0.1502 - val_accuracy: 0.8061 - val_loss: 0.6816
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 861ms/step - accuracy: 0.9735 - loss: 0.0782 - val_accuracy: 0.7896 - val_loss: 0.7050
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 262s 860ms/step - accuracy: 0.9751 - loss: 0.0720 - val_accuracy: 0.7929 - val_loss: 0.7056
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 160ms/step - accuracy: 0.8089 - loss: 0.5036
95/95 ━━━━━━━━━━━━━━━━━━━━ 15s 160ms/step
Test accuracy: 0.8088961839675903
                  precision    recall  f1-score   support

      claude-2.1       0.94      0.84      0.89      1116
      gpt-4-0613       0.74      0.90      0.81      1233
llama-2-70b-chat       0

In [205]:
applyRNN(X_train_Full, X_test_Full, Y_train, Y_test)

--------------------------------------
RNN with:
Hidden dense layer: False
Output dimensions: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 876s 3s/step - accuracy: 0.5050 - loss: 1.0082 - val_accuracy: 0.6249 - val_loss: 0.8279
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 898s 3s/step - accuracy: 0.6844 - loss: 0.6846 - val_accuracy: 0.6686 - val_loss: 0.7035
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 895s 3s/step - accuracy: 0.8107 - loss: 0.4631 - val_accuracy: 0.7509 - val_loss: 0.5931
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 905s 3s/step - accuracy: 0.8970 - loss: 0.2856 - val_accuracy: 0.7608 - val_loss: 0.6133
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 902s 3s/step - accuracy: 0.9345 - loss: 0.2013 - val_accuracy: 0.7555 - val_loss: 0.6705
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 899s 3s/step - accuracy: 0.9612 - loss: 0.1268 - val_accuracy: 0.7715 - val_loss: 0.7295
95/95 ━━━━━━━━━━━━━━━━━━━━ 55s 583ms/step - accuracy: 0.7499 - loss: 0.5948
95/95 ━━━━━━━━━━━━━━━━━━━━ 54s 571ms/step
Test accuracy: 0.7499176263809204
                  precision    recall  f1-score   support

      claude-2.1       0.8

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 902s 3s/step - accuracy: 0.5602 - loss: 0.9261 - val_accuracy: 0.6410 - val_loss: 0.7842
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 897s 3s/step - accuracy: 0.7821 - loss: 0.5470 - val_accuracy: 0.6587 - val_loss: 0.7991
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 906s 3s/step - accuracy: 0.8796 - loss: 0.3244 - val_accuracy: 0.6982 - val_loss: 0.9079
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 912s 3s/step - accuracy: 0.9413 - loss: 0.1683 - val_accuracy: 0.6748 - val_loss: 1.0636
95/95 ━━━━━━━━━━━━━━━━━━━━ 50s 530ms/step - accuracy: 0.6376 - loss: 0.7719
95/95 ━━━━━━━━━━━━━━━━━━━━ 50s 530ms/step
Test accuracy: 0.6375617980957031
                  precision    recall  f1-score   support

      claude-2.1       0.82      0.74      0.78      1116
      gpt-4-0613       0.63      0.55      0.59      1233
llama-2-70b-chat       0.45      0.62      0.52       686

        accuracy                           0.64      3035
       macro avg       0.63      0.64      0.63    

...

304/304 ━━━━━━━━━━━━━━━━━━━━ 924s 3s/step - accuracy: 0.5828 - loss: 0.8769 - val_accuracy: 0.6809 - val_loss: 0.7784
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 885s 3s/step - accuracy: 0.7442 - loss: 0.6202 - val_accuracy: 0.6645 - val_loss: 0.7693
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 829s 3s/step - accuracy: 0.8555 - loss: 0.3950 - val_accuracy: 0.6867 - val_loss: 0.8343
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 799s 3s/step - accuracy: 0.9373 - loss: 0.1839 - val_accuracy: 0.6822 - val_loss: 1.0944
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 812s 3s/step - accuracy: 0.9665 - loss: 0.0999 - val_accuracy: 0.6768 - val_loss: 1.2589
95/95 ━━━━━━━━━━━━━━━━━━━━ 44s 460ms/step - accuracy: 0.6639 - loss: 0.7649
95/95 ━━━━━━━━━━━━━━━━━━━━ 43s 449ms/step
Test accuracy: 0.6639209389686584
                  precision    recall  f1-score   support

      claude-2.1       0.76      0.69      0.73      1116
      gpt-4-0613       0.64      0.68      0.66      1233
llama-2-70b-chat       0.57      0.58  

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 572s 2s/step - accuracy: 0.5194 - loss: 0.9658 - val_accuracy: 0.6727 - val_loss: 0.7516
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 552s 2s/step - accuracy: 0.7550 - loss: 0.5927 - val_accuracy: 0.7423 - val_loss: 0.6199
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 555s 2s/step - accuracy: 0.8881 - loss: 0.3155 - val_accuracy: 0.7670 - val_loss: 0.6040
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 555s 2s/step - accuracy: 0.9414 - loss: 0.1763 - val_accuracy: 0.7670 - val_loss: 0.7066
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 549s 2s/step - accuracy: 0.9619 - loss: 0.1197 - val_accuracy: 0.7526 - val_loss: 0.8149
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 545s 2s/step - accuracy: 0.9718 - loss: 0.0928 - val_accuracy: 0.7509 - val_loss: 0.9124
95/95 ━━━━━━━━━━━━━━━━━━━━ 35s 366ms/step - accuracy: 0.7506 - loss: 0.6311
95/95 ━━━━━━━━━━━━━━━━━━━━ 35s 369ms/step
Test accuracy: 0.7505766153335571
                  precision    recall  f1-score   support

      claude-2.1       0.8

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 554s 2s/step - accuracy: 0.5847 - loss: 0.8812 - val_accuracy: 0.6274 - val_loss: 0.8527
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 555s 2s/step - accuracy: 0.7431 - loss: 0.6135 - val_accuracy: 0.6760 - val_loss: 0.7449
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 556s 2s/step - accuracy: 0.8680 - loss: 0.3657 - val_accuracy: 0.6772 - val_loss: 0.8393
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 545s 2s/step - accuracy: 0.9364 - loss: 0.1865 - val_accuracy: 0.6953 - val_loss: 1.0027
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 544s 2s/step - accuracy: 0.9700 - loss: 0.0972 - val_accuracy: 0.6830 - val_loss: 1.4091
95/95 ━━━━━━━━━━━━━━━━━━━━ 35s 365ms/step - accuracy: 0.6768 - loss: 0.7272
95/95 ━━━━━━━━━━━━━━━━━━━━ 36s 375ms/step
Test accuracy: 0.6767709851264954
                  precision    recall  f1-score   support

      claude-2.1       0.82      0.76      0.79      1116
      gpt-4-0613       0.61      0.77      0.68      1233
llama-2-70b-chat       0.57      0.38  